# EDA Framework

For future datasets, remember this sequence:

**Understand → Validate → Clean → Describe → Visualize → Compare → Investigate → Explain → Recommend**

A data analyst should not stop at *"sales decreased."* Continue asking:

**Where? → When? → Which segment/product/customer? → What variables are associated with it? → Is it an anomaly or recurring pattern? → What should the business investigate or do next?**



## Step 1 — Import libraries

In [1]:
import pandas as pd
import plotly.express as px

## Step 2 — Load the dataset


In [2]:
df = pd.read_csv(r"C:\Users\yousi\Downloads\listings.csv\listings.csv")
df.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,261065,https://www.airbnb.com/rooms/261065,20260624160311,2026-06-24,city scrape,Westboro Village Executive Suite,A tastefully decorated and well equipped upper...,NaN,https://a0.muscache.com/pictures/airflow/Hosti...,1369632,...,4.95,4.95,4.91,STR-825-691,NaN,1,1,0,0,0.75
1,290033,https://www.airbnb.com/rooms/290033,20260624160311,2026-06-24,city scrape,Rural charm close to the city,Close to Strathmere and other wedding venues. ...,NaN,https://a0.muscache.com/pictures/3016522/1e453...,415201,...,4.95,5.00,4.91,STR-834-339,NaN,1,1,0,0,0.14
2,490182,https://www.airbnb.com/rooms/490182,20260624160311,2026-06-24,city scrape,Cozy Basement Studio Old Ottawa East,NaN,NaN,https://a0.muscache.com/pictures/5a7e6363-5b91...,2401179,...,5.00,4.92,4.83,NaN,NaN,1,0,1,0,0.12
3,674799,https://www.airbnb.com/rooms/674799,20260624160311,2026-06-24,city scrape,Up to 3-bdrm with privacy - quiet & safe,"Friendly, responsive hosting, convenient locat...",NaN,https://a0.muscache.com/pictures/39672e5f-36f6...,2058676,...,4.81,4.73,4.65,STR-827-424,NaN,1,1,0,0,0.73
4,682632,https://www.airbnb.com/rooms/682632,20260624160311,2026-06-24,city scrape,"Adorable 2 bdrm, Central Ottawa PD1","This an adorable property with two bedrooms, f...",NaN,https://a0.muscache.com/pictures/3dac520b-19e2...,3201295,...,4.91,4.91,4.73,NaN,NaN,8,4,4,0,0.08



**Ask:** What does one row represent? How many records and columns do we have?

In [3]:
print(f"Rows: {df.shape[0]:,}  Columns: {df.shape[1]}")
print("Each row = one Airbnb listing (one row per `id`).")


Rows: 2,727  Columns: 90
Each row = one Airbnb listing (one row per `id`).


## Step 3 — Understand structure and data types

**Ask:** Are numeric fields numeric? Is the date actually stored as a date?


In [4]:
df.dtypes.value_counts()
df[['price','last_scraped','first_review','last_review','host_is_superhost',
    'instant_bookable','has_availability','bathrooms_text','license']].dtypes

price                    str
last_scraped             str
first_review             str
last_review              str
host_is_superhost        str
instant_bookable     float64
has_availability         str
bathrooms_text           str
license                  str
dtype: object

**Convert** data to proper type

In [5]:
date_cols = ['last_scraped','first_review','last_review','calendar_last_scraped']
for c in date_cols:
    df[c] = pd.to_datetime(df[c], errors='coerce')

df['price'] = pd.to_numeric(df['price'].astype(str).str.replace(r'[\$,]', '', regex=True), errors='coerce')
df['price_quote_total_price'] = pd.to_numeric(df['price_quote_total_price'], errors='coerce')

for c in ['host_is_superhost','host_has_profile_pic','host_identity_verified','has_availability']:
    df[c] = df[c].map({'t': True, 'f': False})



## Step 4 — Check missing values

**Ask:** Which columns have missing values, and what percentage is missing?


In [6]:
missing = pd.DataFrame({'missing_count': df.isna().sum(), 'missing_pct': (df.isna().mean()*100).round(1)})
missing = missing[missing.missing_count > 0].sort_values('missing_pct', ascending=False)
missing


,missing_count,missing_pct
neighborhood_overview,2727,100.0
host_since,2727,100.0
host_acceptance_rate,2727,100.0
host_response_rate,2727,100.0
host_response_time,2727,100.0
host_neighbourhood,2727,100.0
host_total_listings_count,2727,100.0
host_verifications,2727,100.0
host_thumbnail_url,2727,100.0
neighbourhood,2727,100.0


## Step 5 — Check duplicates and uniqueness

**Ask:** Are there duplicated rows? 


In [7]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicated listing ids:", df["id"].duplicated().sum())

Fully duplicated rows: 0
Duplicated listing ids: 0


## Step 6 — Clean obvious data-quality issues

For practice, we will:
- remove exact duplicate rows;
- fill missing categorical values with `"Unknown"`;

In [8]:
before = len(df)
df = df.drop_duplicates()
obj_cols = df.select_dtypes(include='object').columns
df[obj_cols] = df[obj_cols].fillna("Unknown")

C:\Users\yousi\AppData\Local\Temp\ipykernel_14160\2502234713.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(include='object').columns


## Step 7 — Descriptive statistics

**Ask:** What are the typical values, ranges, and variability?


In [9]:
FULLY_EMPTY = ['host_since','host_response_time','host_acceptance_rate','host_response_rate',
               'host_verifications','host_total_listings_count','neighbourhood','host_neighbourhood',
               'calendar_updated','instant_bookable','neighbourhood_group_cleansed','neighborhood_overview']

numeric_df = df.select_dtypes(include='number').drop(columns=[c for c in FULLY_EMPTY if c in df.columns], errors='ignore')
numeric_df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
id,2727.0,8.653666e+17,6.115413e+17,2.610650e+05,5.159270e+07,9.935834e+17,1.397810e+18,1.714089e+18
scrape_id,2727.0,2.026062e+13,0.000000e+00,2.026062e+13,2.026062e+13,2.026062e+13,2.026062e+13,2.026062e+13
host_id,2727.0,3.686802e+15,7.852829e+16,1.104060e+05,6.094450e+07,1.898480e+08,4.614276e+08,1.697240e+18
host_profile_id,2727.0,1.475272e+18,3.251819e+16,1.462510e+18,1.463376e+18,1.468377e+18,1.470143e+18,1.709608e+18
hosts_time_as_user_years,2727.0,6.880000e+00,3.730000e+00,0.000000e+00,4.000000e+00,8.000000e+00,1.000000e+01,1.600000e+01
hosts_time_as_user_months,2727.0,5.770000e+00,3.660000e+00,0.000000e+00,2.000000e+00,6.000000e+00,9.000000e+00,1.100000e+01
hosts_time_as_host_years,2727.0,4.550000e+00,3.530000e+00,0.000000e+00,2.000000e+00,4.000000e+00,8.000000e+00,1.400000e+01
hosts_time_as_host_months,2727.0,5.580000e+00,3.460000e+00,0.000000e+00,2.000000e+00,6.000000e+00,9.000000e+00,1.100000e+01
host_thumbnail_url,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
host_listings_count,2727.0,7.210000e+00,1.560000e+01,1.000000e+00,1.000000e+00,2.000000e+00,5.000000e+00,1.520000e+02


In [10]:
cat_df = df.select_dtypes(include='object')
cat_df.describe().T[['count','unique','top','freq']]

C:\Users\yousi\AppData\Local\Temp\ipykernel_14160\2854301159.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_df = df.select_dtypes(include='object')


,count,unique,top,freq
listing_url,2727,2727,https://www.airbnb.com/rooms/261065,1
source,2727,2,city scrape,2402
name,2727,2695,Room in Ottawa,5
description,2727,2424,Unknown,31
picture_url,2727,2646,https://a0.muscache.com/pictures/airflow/Hosti...,7
host_url,2727,1616,https://www.airbnb.com/users/show/85297148,28
host_profile_url,2727,1616,https://www.airbnb.com/users/profile/146702866...,28
host_name,2727,1202,Seynabou,30
host_location,2727,47,"Ottawa, Canada",1783
host_about,2727,698,Unknown,1473


## Step 8 — Explore categorical variables

**Ask:** What are the most common segments, regions, categories, and shipping modes?


In [11]:
categorical_cols = ['room_type', 'property_type', 'neighbourhood_cleansed', 'source', 'has_availability']

**Visualize** the categorical columns

In [12]:

for col in categorical_cols:

    value_counts = (
        df[col]
        .value_counts()
        .head(12)
        .reset_index()
    )

    value_counts.columns = [col, "Count"]

    fig = px.bar(
        value_counts,
        x=col,
        y="Count",
        text_auto=True,
        title=f"{col.title()} - Frequency Distribution"
    )

    fig.show()

## Step 9 — Explore numerical distributions

Histograms help identify skewness, concentration, and unusual values.


In [13]:
numbers = ['price','accommodates','bedrooms','bathrooms','beds','minimum_nights',
           'availability_365','number_of_reviews','review_scores_rating','reviews_per_month',
           'estimated_revenue_l365d','estimated_occupancy_l365d','host_listings_count']

**Visualize** the numerical columns

In [14]:
for col in numbers:

    fig = px.histogram(df,x=col,
        nbins=30,
        title=f"Distribution of {col.title()}",
        labels={
            col: col.title(),
            "count": "Frequency"
        })

    fig.show()

## Step 10 — Detect outliers using IQR

The common IQR rule defines potential outliers as values below:

`Q1 - 1.5 × IQR`

or above:

`Q3 + 1.5 × IQR`


In [15]:
rows = []
for col in numbers:
    s = df[col].dropna()
    q1, q3 = s.quantile(.25), s.quantile(.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_out = ((s < lo) | (s > hi)).sum()
    rows.append([col, round(lo,2), round(hi,2), n_out, round(100*n_out/len(s),1)])

outlier_table = pd.DataFrame(rows, columns=['column','lower_bound','upper_bound','n_outliers','pct_outliers'])
outlier_table.sort_values('pct_outliers', ascending=False)

,column,lower_bound,upper_bound,n_outliers,pct_outliers
3,bathrooms,0.25,2.25,322,14.2
12,host_listings_count,-5.00,11.00,366,13.4
7,number_of_reviews,-98.00,166.00,238,8.7
0,price,-111.73,391.56,210,8.5
1,accommodates,-1.00,7.00,230,8.4
10,estimated_revenue_l365d,-38260.88,63768.12,185,7.4
9,reviews_per_month,-3.18,6.01,126,5.8
8,review_scores_rating,4.30,5.42,116,5.3
5,minimum_nights,-42.50,73.50,44,1.6
4,beds,-2.00,6.00,32,1.4


### Visualize outliers with a box plot

In [16]:
for col in numbers:
    fig = px.box(
        df,
        y=col,
        title=f"Box Plot: {col.title()}"
    )

    fig.show()

## Step 11 — Analyze relationships between numerical variables

**Ask:** use correlation


In [17]:
corr = df[numbers].corr().round(2)
corr

fig = px.imshow(corr, text_auto=True, color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                 title="Correlation Matrix — Key Numeric Variables")
fig.show()

In [18]:
fig = px.scatter_matrix(
    df,
    dimensions=numbers,
    title="Scatter Matrix of Numeric Variables"
)

fig.show()

## Step 12 — Compare performance across groups

**Ask:** use Groupby

In [19]:
room_summary = df.groupby('room_type').agg(
    listings=('id','count'), avg_price=('price','mean'), median_price=('price','median'),
    avg_rating=('review_scores_rating','mean'), avg_avail_365=('availability_365','mean')
).round(2).sort_values('listings', ascending=False)
room_summary

,listings,avg_price,median_price,avg_rating,avg_avail_365
room_type,,,,,
Entire home/apt,1664,224.82,160.91,4.81,209.69
Private room,1046,91.63,77.00,4.71,234.66
Hotel room,15,258.53,219.00,4.42,347.27
Shared room,2,113.78,113.78,4.88,197.00


In [20]:
df.groupby('neighbourhood_cleansed').agg(
    listings=('id','count'), avg_price=('price','mean'), avg_rating=('review_scores_rating','mean')
).round(2).sort_values('listings', ascending=False).head(10)

,listings,avg_price,avg_rating
neighbourhood_cleansed,,,
Rideau-Vanier,434,160.30,4.62
Somerset,219,159.18,4.71
Kitchissippi,164,176.94,4.86
Barrhaven,154,159.58,4.73
Gloucester-South Nepean,151,197.73,4.88
Rideau-Goulbourn,140,207.63,4.80
Rideau-Rockcliffe,136,146.97,4.79
Capital,124,205.87,4.87
College,113,165.32,4.84


In [21]:
df.groupby('host_is_superhost').agg(
    listings=('id','count'), avg_price=('price','mean'), avg_rating=('review_scores_rating','mean'),
    avg_reviews_per_month=('reviews_per_month','mean')
).round(2)

,listings,avg_price,avg_rating,avg_reviews_per_month
host_is_superhost,,,,
False,1589,155.89,4.66,1.46
True,1138,196.35,4.88,2.13


In [22]:
plot_df = room_summary.reset_index().sort_values('avg_price', ascending=False)
fig = px.bar(plot_df, x='room_type', y='avg_price', title="Average Price by Room Type",
             text='avg_price')
fig.show()

## Step 13 — Time-series EDA

**Ask:** what is trend over time? Are some months stronger than others?


In [23]:
ts = df.dropna(subset=['first_review']).copy()
ts['review_month'] = ts['first_review'].dt.to_period('M').dt.to_timestamp()
monthly_new_reviews = ts.groupby('review_month').size().reset_index(name='count')

fig = px.line(monthly_new_reviews, x='review_month', y='count',
              title="Listings' First-Review Month Over Time")
fig.show()

In [24]:
ts['review_calmonth'] = ts['first_review'].dt.month
seasonality = ts.groupby('review_calmonth').size().reset_index(name='count')

fig = px.bar(seasonality, x='review_calmonth', y='count',
             title="Seasonality: First Reviews by Calendar Month (all years combined)")
fig.update_layout(xaxis=dict(dtick=1, title="Month"))
fig.show()

## Step 14 — Find the best and worst performers

A useful EDA habit is to inspect both extremes rather than only averages.


In [25]:
df.nlargest(10, 'estimated_revenue_l365d')[['id','name','neighbourhood_cleansed','room_type','price','estimated_revenue_l365d','review_scores_rating']]

,id,name,neighbourhood_cleansed,room_type,price,estimated_revenue_l365d,review_scores_rating
2272,1516010064385111265,Retreat & Recharge | Private Pool + Hot Tub Oasis,College,Entire home/apt,1347.00,290952.0,4.81
938,702299454842917257,"Luxury 10 Bedroom Mansion w/HotTub, Pool Table...",River,Entire home/apt,911.00,232305.0,4.88
1282,955352002212818262,"Luxury Palace W/Hot-tub, Arcade, & 11 beds-Air...",Gloucester-South Nepean,Entire home/apt,911.00,232305.0,4.87
728,53302297,Waterfront Retreat Sauna Hot Tub Kayak Canoe Fish,West Carleton-March,Entire home/apt,924.35,227390.0,4.92
438,36118178,5bdrm Sauna/Hottub/Arcades/Close to DT and Beach,Bay,Entire home/apt,899.00,210366.0,4.87
2086,1426871532084462583,Luxury 6BR Home | 8 Beds 3.5Bath,Osgoode,Entire home/apt,728.07,185658.0,5.00
2207,1492226375298215610,"Pool, Game Room, Hot Tub, Theatre Room",West Carleton-March,Entire home/apt,825.33,178271.0,5.00
831,598440933597251505,6 bdrm Calm&Private Farm In Ottawa: The Farm Haus,Rideau-Goulbourn,Entire home/apt,725.00,169650.0,4.98
944,705271948873559154,"Brand New Luxury Home W/8Beds, Hot-tub, Pool T...",Gloucester-Southgate,Entire home/apt,626.00,159630.0,4.91
1012,761569055889237974,Entire House: 5BR + Optional 2BR Basement | Ka...,Kanata South,Entire home/apt,626.00,159630.0,4.88


In [26]:
reviewed = df[df['number_of_reviews'] >= 10]
reviewed.nlargest(10, 'review_scores_rating')[['id','name','neighbourhood_cleansed','room_type','price','review_scores_rating','number_of_reviews']]

,id,name,neighbourhood_cleansed,room_type,price,review_scores_rating,number_of_reviews
1,290033,Rural charm close to the city,Rideau-Goulbourn,Entire home/apt,224.81,5.0,22
2,490182,Cozy Basement Studio Old Ottawa East,Capital,Private room,67.13,5.0,12
43,6512008,"Westboro: Petit déj' compris, tranquille - Chatte",Kitchissippi,Private room,120.00,5.0,25
56,7782901,Room w/ bed & desk in an Ottawa homestay!,Rideau-Vanier,Private room,40.43,5.0,24
58,7901417,Bed & desk in ByWard Homestay near uOttawa,Rideau-Vanier,Private room,NaN,5.0,52
87,11117632,Luxury Modern Townhouse On A Traffic-Free Street,Barrhaven,Entire home/apt,29.84,5.0,12
100,12421106,2Bed/2Baths in Ottawa Down-town Core.,Somerset,Entire home/apt,234.14,5.0,16
141,15639698,"Private room, art filled home in trendy hood",Kitchissippi,Private room,79.19,5.0,13
157,16348342,Clean & Modern Living in Little Italy & Chinatown,Somerset,Entire home/apt,201.00,5.0,257
192,18757065,"Bright Room (L-desk), river near",Orleans,Private room,34.80,5.0,16


In [27]:
reviewed.nsmallest(10, 'review_scores_rating')[['id','name','neighbourhood_cleansed','room_type','price','review_scores_rating','number_of_reviews']]

,id,name,neighbourhood_cleansed,room_type,price,review_scores_rating,number_of_reviews
303,26178074,Tiny Room Across uOttawa Steps To Downtown Ottawa,Rideau-Vanier,Private room,57.42,3.60,10
576,45228088,Room Across UOttawa Steps TO Downtown Ottawa,Rideau-Vanier,Private room,54.65,3.91,11
146,15972312,Little Jewel in Central Ottawa 3bd,River,Entire home/apt,98.66,3.95,174
990,747070099643623624,Lovely Studio in Downtown Ottawa,Somerset,Entire home/apt,69.00,4.00,10
1126,861813979357491666,Cozy spacious bedroom in Ottawa,Barrhaven,Private room,66.39,4.00,18
1855,1274158587166664273,Serene & Homey room (R2),Barrhaven,Private room,37.00,4.00,17
642,48661104,The Park,River,Entire home/apt,104.27,4.04,79
694,51961737,Cozy Private Bedroom!,Barrhaven,Private room,NaN,4.05,19
1264,946470235813139098,Lovely 4-Bedroom Ottawa,Rideau-Vanier,Entire home/apt,NaN,4.11,18
616,47411380,Cozy One Bedroom w/ Parking - Mins to Downtown!,Kitchissippi,Entire home/apt,NaN,4.12,84


## Step 15 — Business questions

Try answering these from the analysis above:

1. Which region generates the highest sales?
2. Which region generates the highest profit?
3. Which category generates the highest sales?
4. Which category is most profitable?
5. Does higher discount appear to reduce profitability?
6. Which orders are unusually large?
7. Are high-sales orders always high-profit orders?
8. Which month has the highest sales?
9. Which customer segment contributes the most revenue?
10. What data-quality issues did we discover?


1. Rideau-Vanier leads by about $7.06M in estimated annual revenue
2. West Carleton-March at an average of $384/night
3. Entire home/apt at $39.4M total
4. Hotel room at $259/night average
5. There's no discount field but superhost status is the closest. Superhosts average $29,629 in annual revenue per listing vs $10,854 for non-superhosts, nearly 3x despite being a minority of listings
6. bathrooms (14.2% of listings) and host_listings_count (13.4%) have the most, followed by number_of_reviews (8.7%) and price (8.5%).
7. Not exactly, but there's a positive relationship. Price correlates with estimated revenue at r = 0.57. Number of reviews correlates with revenue even more weakly on its own (r = 0.42). So revenue is driven by a mix of price and demand, not price alone.
8. May with 242 listings.
9. Superhosts contribute $31.7M total vs $15.4M for non-superhosts.
10. 12 columns were 100% empty (host_since, host_response_rate, instant_bookable, etc.). price was stored as text with a $ prefix. Several numeric fields (host_is_superhost, has_availability) were stored as t/f strings. About 20% of listings were missing review-score data (no reviews yet), and bedrooms was missing in 29% of rows.